# Day 5 补充：从字段到矩阵、参数与内存

先读 `docs/DAY05_ARCHITECTURES.md` 的 2.1.5 与 2.1.9–2.1.11。
新增第0节区分投影与拆头；两次残差分叉图见 [统一符号规范](../docs/NOTATION.md)。
本实验只做配置算术，不加载 0.6B 权重，不分配4096×4096的score。
每一步先预测再执行；本实验属于Day 5，当前学习位置见 [进度记录](../docs/LEARNING_PROGRESS.md)。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/config_math.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from learning.config_math import inspect_config
from reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B
cfg = dict(QWEN_CONFIG_06_B)
report = inspect_config(cfg, tokens=4096)
print(report['dimensions'])

## 0. 为什么D不一定等于E/Hq：先投影，再拆头

Linear有权重，可将E维输入变成Hq×D维输出；view/reshape没有权重，只对输出重新分组。
恒成立的是`D=Q投影宽度/Hq`，不是`D=输入宽度E/Hq`。
假设E=4、Hq=2、D=3，输入`[a,b,c,d]=[1,2,3,4]`。
人为选6个线性组合`[a+b,b+c,c+d,a-c,b-d,a+b+c+d]`，先手算，再执行。
这些是输入的新组合，不会凭空增加独立信息；不是把4个数直接reshape成6个。

In [ ]:
import torch

toy_x = torch.tensor([[[1., 2., 3., 4.]]])
toy_weight = torch.tensor([
    [1., 1., 0., 0.],
    [0., 1., 1., 0.],
    [0., 0., 1., 1.],
    [1., 0., -1., 0.],
    [0., 1., 0., -1.],
    [1., 1., 1., 1.],
])
toy_q = toy_x @ toy_weight.T
toy_heads = toy_q.view(1, 1, 2, 3).transpose(1, 2)
torch.testing.assert_close(toy_q, torch.tensor([[[3., 5., 7., -2., -2., 10.]]]))
assert toy_heads.shape == (1, 2, 1, 3)
assert toy_heads.numel() == toy_q.numel() == 6
assert toy_x.numel() == 4
torch.testing.assert_close(toy_heads[0, 0, 0], torch.tensor([3., 5., 7.]))
torch.testing.assert_close(toy_heads[0, 1, 0], torch.tensor([-2., -2., 10.]))
print('输入:', toy_x.tolist(), tuple(toy_x.shape))
print('权重shape [out,in]:', tuple(toy_weight.shape))
print('投影:', toy_q.tolist(), tuple(toy_q.shape))
print('拆头:', toy_heads.tolist(), tuple(toy_heads.shape))
try:
    toy_x.view(1, 1, 2, 3)
except RuntimeError:
    print('预期负例：不能把4个输入数直接view成2*3=6个数。')
else:
    raise AssertionError('A view must preserve element count.')

### 对照真实类的两种构造方式

复用仓库GroupedQueryAttention，仅建立很小的随机模块并调用Q投影，不加载模型或checkpoint。
固定E=8、Hq=2、Hkv=1。先预测：head_dim=None与显式head_dim=6的W_query形状各是什么？
模块随机种子放在fork_rng内，避免影响后续单元格。此处只验证投影和拆头，不是完整Attention质量/速度实验。

In [ ]:
from reasoning_from_scratch.qwen3 import GroupedQueryAttention

with torch.random.fork_rng(devices=[]):
    torch.manual_seed(123)
    small_x = torch.randn(1, 3, 8)
    for setting, expected_dim in ((None, 4), (6, 6)):
        module = GroupedQueryAttention(8, 2, 1, head_dim=setting, dtype=torch.float32)
        flat = module.W_query(small_x)
        split = flat.view(1, 3, 2, module.head_dim).transpose(1, 2)
        assert module.head_dim == expected_dim
        assert module.W_query.weight.shape == (2 * expected_dim, 8)
        assert module.out_proj.weight.shape == (8, 2 * expected_dim)
        assert split.shape == (1, 2, 3, expected_dim)
        print('head_dim=', setting, '实际D=', module.head_dim,
              'W_query=', tuple(module.W_query.weight.shape),
              '投影=', tuple(flat.shape), '拆头=', tuple(split.shape))
print('Qwen3配置：E=1024,Hq=16,D=128，Q宽度=2048；E/Hq=64不是该显式D。')

## 1. 2048维输出不等于2048个参数

`nn.Linear(in,out,bias=False)` 的权重为 `[out,in]`，计算 `x @ W.T`。
先算 Q、K、V、out 和 gate/up/down 的 shape，再看输出。
本表不含 Norm、Embedding、LM Head，不能称为模型总参数。

In [ ]:
for name, shape in report['weight_shapes_out_in'].items():
    print(f'{name:12} shape={shape}, parameters={report["parameters"][name]:,}')
print('单层主要投影参数:', f'{report["projection_parameters_per_layer"]:,}')
assert report['weight_shapes_out_in']['W_query'] == [2048, 1024]
assert report['weight_shapes_out_in']['down_proj'] == [1024, 3072]

## 2. 三种不同的内存

假设权重、Cache、score 每元素都是2字节。
score 是单层临时张量，Cache 是所有层的历史状态；不是直接相加就得到进程峰值。
T从4096翻倍到8192，先预测权重、KV、score分别变成几倍。

In [ ]:
larger = inspect_config(cfg, tokens=8192)
for key in ('projection_weight_bytes_all_layers', 'kv_bytes_all_layers', 'prefill_score_bytes_one_layer'):
    before, after = report[key], larger[key]
    print(f'{key}: {before / 2**20:.1f} -> {after / 2**20:.1f} MiB; ratio={after / before:g}')
assert report['kv_bytes_all_layers'] == 448 * 2**20
assert report['prefill_score_bytes_one_layer'] == 512 * 2**20
print('适用范围:', report['scope'])

## 3. 一次改一个字段

先预测 Hkv 从8改4、F从3072改4096分别影响哪些投影。
这里只推导新结构，不能把原checkpoint无条件加载进改变shape的新模型。

In [ ]:
for field, value in (('n_kv_groups', 4), ('hidden_dim', 4096)):
    changed = inspect_config(dict(cfg, **{field: value}), tokens=4096)
    print('\n修改', field, '=', value)
    for name, shape in changed['weight_shapes_out_in'].items():
        if shape != report['weight_shapes_out_in'][name]:
            print(name, report['weight_shapes_out_in'][name], '->', shape)
    print('KV MiB:', changed['kv_bytes_all_layers'] / 2**20)
    print('sharing ratio:', changed['sharing_ratio'])

## 验收与下一步

- 为什么D=Q投影宽度/Hq成立，但D=E/Hq不总成立？
- Q投影为什么需要 2048×1024 个权重，而非2048个？
- 为什么FFN的主要参数是3EF，而不是2EF？
- T翻倍，权重不变、Cache两倍、prefill score四倍，分别为什么？
- 为什么不能把所有层Cache和单层score都再乘28？

### 参考答案

1. 被拆分的是Q投影结果，不是原始输入。仅当Q宽度=E时才有D=E/Hq。
2. 每个输出特征有1024个输入系数，2048个输出共有2048组系数。
3. SwiGLU包含gate、up、down三套权重，各有EF个参数；忽略bias时总计3EF。
4. 权重由模型结构决定；Cache每位置新增一份K/V；完整Prefill的Q和K位置轴同时翻倍，score变4倍。
5. Cache公式已包含28层，不能再次乘；score公式只指单层临时数组，不能据此假定28层完整score同时驻留。

解释清楚后，继续 `learning/day06_online_softmax.ipynb`。